# Refit a transit with SciPy

A reviewed notebook for the Gravitas compute page. It reads one `gravitas.observation/1` time series, handed to it as the text `gravitas_input`, and its one setting as the text `gravitas_params`. The example is the TESS light curve of HD 209458 that the Observatory's fit panel fits. The notebook fits the same model to the same points by a different route, so that what differs can be traced to the route and not to the model.

The model is the fit panel's: a circular orbit, quadratic limb darkening in Kipping's `q1, q2`, the light blocked computed by the same direct quadrature (32 nodes), each point the mean of five instants across its 20-minute exposure, and a baseline `f0` solved exactly at every step. Seven numbers are fitted: the epoch `t0`, the period `P`, the planet's size `k = Rp/R*`, the orbit's size `aRs = a/R*`, the impact parameter `b`, and `q1`, `q2`. It was written again here in NumPy and agrees with the application's to the last digit of a double.

What it does, in order:

1. **Start.** `scipy.ndimage.label` groups the points well below the median into transits; a straight line through their mid-times gives the period and epoch, and the depth gives `k`. Three impact parameters (0.1, 0.45, 0.75) give three starts, as the panel's own search does.
2. **Fit.** `scipy.optimize.least_squares`, the bounded trust-region method, on the weighted residuals. This is a different optimizer from the panel's Levenberg-Marquardt with an active set.
3. **Uncertainty from the covariance.** The inverse of `J^T J` at the minimum, scaled by the reduced chi-square when it is above 1 (the panel's `sigmaScaled`).
4. **Uncertainty from the profile.** For `k` and `b`: hold the parameter at a value, refit every other one, and find with `scipy.optimize.brentq` where chi-square (scaled) has risen by 1. This is the panel's profile interval, found by root-finding where the panel interpolates between grid points.

It returns an `envelope` (the five fitted numbers with scaled sigmas, the two profile intervals, the reduced chi-square and the number of points) and a `table` of the folded light curve in 16-minute bins. The panel's own result for the same light curve is the thing to compare it with: the two should agree closely, and where they do not, the reason is in the list above.

What is not covered: this is a white-noise fit with the error bars inflated by one constant. The residuals of this light curve are correlated in time, which the panel's red-noise factor (`sigmaRed`) counts and this notebook does not, so its sigmas are the panel's `sigmaScaled`, not its `sigmaRed`. The period and epoch are the light curve's own time (BTJD, TDB).

In [ ]:
# Read the input and the setting.
import json, hashlib
import numpy as np
from scipy import ndimage, optimize

obs = json.loads(gravitas_input)
params = json.loads(gravitas_params)
assert obs["format"] == "gravitas.observation" and obs["formatVersion"] == 1, "this notebook reads gravitas.observation/1"
assert obs["kind"] == "time-series", "this notebook reads a time series"
LIMB = int(round(float(params.get("limb", 1))))
assert LIMB in (0, 1), "limb is 0 (limb darkening held fixed) or 1 (fitted)"
input_digest = hashlib.sha256(gravitas_input.encode("utf-8")).hexdigest()

def column(role, of=None):
    for c in obs["columns"]:
        if c["role"] == role and (of is None or c.get("of") == of):
            return c
    return None

xc = next(c for c in obs["columns"] if c["id"] == obs["time"]["column"])
assert xc["unit"] == "d", "the time column must be in days"
yc = column("value")
ec = column("uncertainty", of=yc["id"])
assert ec is not None, "the light curve needs an uncertainty column"
num = lambda c: np.array([np.nan if v is None else v for v in c["values"]], float)
t, f, e = num(xc), num(yc), num(ec)
masked = np.zeros(len(t), bool)
for m in obs["masks"]:
    masked[m["rows"]] = True
keep = np.isfinite(t) & np.isfinite(f) & np.isfinite(e) & (e > 0) & ~masked
t, f, e = t[keep], f[keep], e[keep]
level = np.median(f)
y, w = f / level, 1.0 / (e / level) ** 2
EXPOSURE = float(np.median(np.diff(t)))
print(f"{len(t)} points from {t.min():.3f} to {t.max():.3f} d ({obs['time']['format']}, {obs['time']['scale']}); exposure {1440 * EXPOSURE:.1f} minutes")

In [ ]:
# The transit model of the Observatory's fit panel, in NumPy (js/inference/transit.js).
def occultation(z, k, u1, u2, n=32):
    z = np.asarray(z, float)
    out = np.zeros_like(z)
    hit = z < 1 + k
    zz = z[hit]
    total = np.pi * (1 - u1 / 3 - u2 / 6)
    F = lambda mu: mu**2 / 2 - u1 * (mu**2 / 2 - mu**3 / 3) - u2 * (mu**2 / 2 - 2 * mu**3 / 3 + mu**4 / 4)
    mu_in = np.sqrt(np.maximum(0.0, 1 - np.minimum(1, k - zz) ** 2))
    inner = np.where(zz < k, 2 * np.pi * (F(1.0) - F(mu_in)), 0.0)
    a = np.abs(zz - k)
    wid = np.minimum(1, zz + k) - a
    s = (np.arange(n) + 0.5) / n
    r = a[:, None] + wid[:, None] * (1 - np.cos(np.pi * s)) / 2
    dr = wid[:, None] * np.pi * np.sin(np.pi * s) / 2 / n
    zs = np.where(zz > 0, zz, 1.0)[:, None]
    arc = 2 * r * np.arccos(np.clip((r * r + zs * zs - k * k) / (2 * r * zs), -1, 1))
    m = 1 - np.sqrt(np.maximum(0.0, 1 - r * r))
    partial = np.where((wid > 0) & (zz > 0), ((1 - u1 * m - u2 * m * m) * arc * dr).sum(axis=1), 0.0)
    out[hit] = np.minimum(1, (inner + partial) / total)
    return out

def transit_flux(t, t0, P, k, aRs, b, q1, q2, exposure, supersample=5, nodes=32):
    u1, u2 = 2 * np.sqrt(q1) * q2, np.sqrt(q1) * (1 - 2 * q2)
    cosi = b / aRs
    sini = np.sqrt(max(0.0, 1 - cosi * cosi))
    arg = np.sqrt(max(0.0, (1 + k) ** 2 - b * b)) / (aRs * sini)
    if not ((1 + k) ** 2 > b * b and arg <= 1):
        return np.ones_like(t)
    half = P / (2 * np.pi) * np.arcsin(arg)
    cycles = (t - t0) / P
    dt = (cycles - np.floor(cycles + 0.5)) * P
    near = np.abs(dt) <= half + exposure / 2
    out = np.ones_like(t)
    if near.any():
        off = exposure * ((np.arange(supersample) + 0.5) / supersample - 0.5)
        tj = t[near][:, None] + off[None, :]
        phi = 2 * np.pi * (tj - t0) / P
        sn, cs = np.sin(phi), np.cos(phi)
        z = aRs * np.sqrt(sn * sn + cosi * cosi * cs * cs)
        occ = occultation(z.ravel(), k, u1, u2, nodes).reshape(z.shape)
        out[near] = np.where(cs > 0, 1 - occ, 1.0).mean(axis=1)
    return out

In [ ]:
# Start: SciPy's labeling finds the transits; a line through their mid-times is the period and epoch.
scatter = 1.4826 * np.median(np.abs(y - np.median(y)))
labels, count = ndimage.label(y < 1 - 5 * scatter)
events = []
for i in range(1, count + 1):
    idx = np.where(labels == i)[0]
    if len(idx) >= 4:
        events.append((t[idx].mean(), 1 - np.median(y[idx])))
print(len(events), "transits found")
if len(events) < 3:
    raise ValueError("fewer than three transits were found: this notebook needs a light curve of a planet that transits several times")
mids = np.array([m for m, _ in events])
number = np.round((mids - mids[0]) / np.min(np.diff(mids))).astype(int)
P0, T00 = np.polyfit(number, mids, 1)
K0 = float(np.sqrt(np.median([d for _, d in events])))
print(f"start: P = {P0:.5f} d, t0 = {T00:.4f}, k = {K0:.3f}")

In [ ]:
# Fit. t0 is fitted as an offset from the start, so every number is of order one and the finite-difference steps are sensible.
NAMES = ["t0", "P", "k", "aRs", "b", "q1", "q2"]
Q_FIXED = (0.36, 0.30)
LO = np.array([-0.1, P0 - 0.05, 0.005, 1.5, 0.0, 0.0, 0.0])
HI = np.array([0.1, P0 + 0.05, 0.5, 200.0, 1.0, 1.0, 1.0])
SCALE = np.array([1e-3, 1e-4, 1e-2, 1.0, 0.1, 0.1, 0.1])
FREE = np.arange(7) if LIMB else np.arange(5)

def residuals(free_values, full, free):
    x = full.copy()
    x[free] = free_values
    shape = transit_flux(t, T00 + x[0], x[1], x[2], x[3], x[4], x[5], x[6], EXPOSURE)
    f0 = (w * y * shape).sum() / (w * shape * shape).sum()
    return (y - f0 * shape) * np.sqrt(w)

def fit(full, free):
    return optimize.least_squares(
        residuals, full[free], args=(full, free), bounds=(LO[free], HI[free]), method="trf",
        x_scale=SCALE[free], xtol=1e-10, ftol=1e-10, gtol=1e-10, jac="3-point", diff_step=1e-6,
    )

best = None
for b0 in (0.1, 0.45, 0.75):
    aRs0 = P0 / (np.pi * 0.12) * np.sqrt(max(0.1, (1 + K0) ** 2 - b0**2))
    full0 = np.array([0.0, P0, K0, aRs0, b0, *Q_FIXED])
    full0 = np.clip(full0, LO + 1e-9, HI - 1e-9)
    r = fit(full0, FREE)
    print(f"start b = {b0}: chi-square {2 * r.cost:.2f} in {r.nfev} evaluations")
    if best is None or r.cost < best.cost:
        best, best_full = r, full0.copy()
        best_full[FREE] = r.x
theta = best_full
chi2 = 2 * best.cost
N = len(t)
dof = N - len(FREE) - 1
reduced = chi2 / dof
print(f"chi-square {chi2:.2f} for {dof} degrees of freedom: reduced {reduced:.3f}")

In [ ]:
# Uncertainty from the covariance, then from the profile of k and b.
J = best.jac
cov = np.linalg.inv(J.T @ J)
sigma = np.sqrt(np.diag(cov)) * np.sqrt(max(1.0, reduced))
scale = max(1.0, reduced)

def profile_chi2(j, value):
    full = theta.copy()
    full[j] = value
    free = np.array([i for i in FREE if i != j])
    r = optimize.least_squares(
        residuals, theta[free], args=(full, free), bounds=(LO[free], HI[free]), method="trf",
        x_scale=SCALE[free], xtol=1e-10, ftol=1e-10, gtol=1e-10, jac="3-point", diff_step=1e-6,
    )
    return 2 * r.cost

def profile_interval(j):
    g = lambda v: (profile_chi2(j, v) - chi2) / scale - 1.0
    ends, bounded = [], False
    for sign in (-1, 1):
        v0, step, v = theta[j], sigma[FREE.tolist().index(j)], theta[j]
        for n in range(1, 13):
            v = float(np.clip(theta[j] + sign * n * step, LO[j], HI[j]))
            if g(v) >= 0 or v in (LO[j], HI[j]):
                break
        if g(v) < 0:
            bounded = True
            ends.append(v)
        else:
            ends.append(float(optimize.brentq(g, min(v0, v), max(v0, v), xtol=1e-3 * step)))
    return ends[0], ends[1], bounded

prof = {j: profile_interval(j) for j in (2, 4)}
for j in (2, 4):
    lo_, hi_, _ = prof[j]
    print(f"{NAMES[j]} = {theta[j]:.5f}: sigma {sigma[FREE.tolist().index(j)]:.5f}, profile {lo_:.5f} to {hi_:.5f}")

In [ ]:
# What the notebook returns: an envelope and a table.
LICENSE = obs["license"]
sig = lambda s, basis="scaled": {"kind": "sigma", "sigma": float(s), "basis": basis}
at = lambda name: sigma[FREE.tolist().index(NAMES.index(name))] if NAMES.index(name) in FREE else None
interval = lambda j: {"kind": "interval", "lo": float(prof[j][0]), "hi": float(prof[j][1]), "level": 0.6827, "basis": "profile"}
quantities = [
    {"id": "t0", "value": float(T00 + theta[0]), "unit": "d", "origin": "fitted", "uncertainty": sig(at("t0"))},
    {"id": "P", "value": float(theta[1]), "unit": "d", "origin": "fitted", "uncertainty": sig(at("P"))},
    {"id": "k", "value": float(theta[2]), "unit": "", "origin": "fitted", "uncertainty": sig(at("k"))},
    {"id": "aRs", "value": float(theta[3]), "unit": "", "origin": "fitted", "uncertainty": sig(at("aRs"))},
    {"id": "b", "value": float(theta[4]), "unit": "", "origin": "fitted", "uncertainty": sig(at("b"))},
    {"id": "k.profile", "value": float(theta[2]), "unit": "", "origin": "fitted", "uncertainty": interval(2)},
    {"id": "b.profile", "value": float(theta[4]), "unit": "", "origin": "fitted", "uncertainty": interval(4)},
    {"id": "chi2.reduced", "value": float(reduced), "unit": "", "origin": "derived", "uncertainty": {"kind": "none"}},
    {"id": "points", "value": float(N), "unit": "count", "origin": "measured", "uncertainty": {"kind": "none"}},
]
warnings = [
    "the sigmas are the covariance's, scaled by the reduced chi-square when it is above 1; the residuals of a real light curve are correlated in time, which this notebook does not count, so they are the panel's sigmaScaled and not its sigmaRed",
    "t0 is in the light curve's own time (" + (obs["time"].get("reference") or obs["time"]["format"]) + ")",
    "the orbit is circular and b is bounded between 0 and 1, as the model of the Observatory's fit panel is",
]
if LIMB:
    warnings.append("q1 and q2 are fitted: " + f"q1 = {theta[5]:.2f} +/- {sigma[5]:.2f}, q2 = {theta[6]:.2f} +/- {sigma[6]:.2f}" + "; with a few transits they are poorly known and trade against b and aRs")
else:
    warnings.append("limb darkening is held at q1 = 0.36, q2 = 0.30, not fitted: the sigmas of b and aRs do not include its uncertainty")
if any(p[2] for p in prof.values()):
    warnings.append("a profile interval reaches a bound of the parameter or is cut off at twelve sigma: it is a lower or upper limit on that side")
envelope = {
    "format": "gravitas.artifact", "formatVersion": 1,
    "id": f"compute:refit-transit:{input_digest[:16]}",
    "made": {"app": "gravitas"},
    "source": {"kind": "analysis", "id": "compute/refit-transit", "version": "1", "digest": input_digest},
    "provenance": {
        "credit": obs["credit"], "license": LICENSE, "retrieved": obs["retrieved"],
        "citations": [c["text"] for c in obs["citations"]],
        "reductions": obs["reductions"] + [
            "Normalized by the median. Transit model of the Observatory's fit panel (circular orbit, quadratic limb darkening in Kipping's q1 and q2, 32-node quadrature, 5 sub-exposures of 20 minutes), baseline solved exactly.",
            "Bounded trust-region least squares (scipy.optimize.least_squares) from three starting impact parameters; the best chi-square is kept.",
            "Sigma from the covariance (J^T J)^-1, scaled by the reduced chi-square when it exceeds 1. Profile interval where chi-square (scaled) rises by 1 with every other free parameter refitted, by scipy.optimize.brentq.",
        ],
    },
    "quantities": quantities,
    "warnings": warnings,
}

# The folded light curve in 16-minute bins, with the fitted model.
shape = transit_flux(t, T00 + theta[0], theta[1], theta[2], theta[3], theta[4], theta[5], theta[6], EXPOSURE)
f0 = (w * y * shape).sum() / (w * shape * shape).sum()
hours = ((t - (T00 + theta[0]) + 0.5 * theta[1]) % theta[1] - 0.5 * theta[1]) * 24
edges = np.arange(-4.0, 4.0 + 1e-9, 16 / 60)
which = np.digitize(hours, edges) - 1
rows = [i for i in range(len(edges) - 1) if (which == i).sum() >= 3]
resid = y - f0 * shape
table = {
    "format": "gravitas.observation", "formatVersion": 1, "kind": "table",
    "id": f"{obs['id']}:folded-fit", "title": f"{obs['title']}: folded at the fitted period, with the fit",
    "object": obs["object"], "facility": obs["facility"], "origin": "model",
    "source": {"kind": "import", "id": obs["id"], "version": obs["source"].get("version")},
    "credit": obs["credit"], "license": LICENSE, "retrieved": obs["retrieved"],
    "citations": obs["citations"],
    "reductions": envelope["provenance"]["reductions"],
    "axes": {"x": "hours", "y": "flux"},
    "columns": [
        {"id": "hours", "name": "hours from mid-transit", "unit": "h", "role": "x", "values": [float(0.5 * (edges[i] + edges[i + 1])) for i in rows]},
        {"id": "flux", "name": "flux (binned)", "unit": "", "role": "value", "values": [float(y[which == i].mean()) for i in rows]},
        {"id": "flux-error", "name": "flux uncertainty", "unit": "", "role": "uncertainty", "of": "flux", "values": [float(resid[which == i].std(ddof=1) / np.sqrt((which == i).sum())) for i in rows]},
        {"id": "model", "name": "fitted model (binned)", "unit": "", "role": "value", "values": [float((f0 * shape)[which == i].mean()) for i in rows]},
        {"id": "residual", "name": "residual (binned)", "unit": "", "role": "value", "values": [float(resid[which == i].mean()) for i in rows]},
    ],
    "masks": [], "annotations": [],
}
gravitas_output = {"envelope": envelope, "table": table}
print(len(rows), "rows returned")